<a href="https://colab.research.google.com/github/GiovanniPasq/agentic-rag-for-dummies/blob/main/notebooks/evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Evaluation

This notebook evaluates the **Agentic RAG for Dummies** system with a small QA dataset built from the Markdown corpus and scored with [RAGAS](https://docs.ragas.io/).

The evaluation uses exactly five metrics:

| Metric | Family | What it measures |
|---|---|---|
| **Answer Accuracy** | [NVIDIA metric](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/nvidia_metrics/#answer-accuracy) | Agreement between the generated answer and the reference answer |
| **Context Relevance** | [NVIDIA metric](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/nvidia_metrics/#context-relevance) | Whether retrieved chunks are pertinent to the user query |
| **Response Groundedness** | [NVIDIA metric](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/nvidia_metrics/#response-groundedness) | Whether response claims are supported by retrieved contexts |
| **Context Precision** | Retrieval metric | Whether relevant retrieved chunks are ranked ahead of irrelevant chunks |
| **Context Recall** | Retrieval metric | Whether the retrieved contexts cover the information needed by the reference answer |

Evaluation flow:

1. Load QA records from `notebooks/data/curated_ragas_qa.json`.
2. Run each question through the agentic RAG system and save its answer plus the actual child/parent tool outputs to `ragas_evaluation_dataset.csv`.
3. Reuse the saved CSV only when it matches the current curated QA dataset, so stale answers are not scored by mistake.
4. Call each RAGAS metric directly with `await scorer.ascore(...)` and average the scores at the end.

RAGAS can also generate synthetic testsets with `TestsetGenerator.generate_with_langchain_docs(...)` as shown in the [official testset generation guide](https://docs.ragas.io/en/stable/getstarted/rag_testset_generation/#choose-your-llm). In this repo, the JSON dataset is easier to inspect, reproduce, and correct. A human-annotated dataset can better capture semantic nuance, acceptable answer variants, and document-specific interpretation rules.

**Single-hop scope.** This notebook evaluates single-hop QA: each question should be answerable from one document area and one retrieval pass. Multi-hop QA requires combining evidence from multiple sections/documents or sequential reasoning steps; it is useful for deeper agent evaluation, but it is intentionally outside this notebook so retrieval relevance, grounding, and answer quality stay easy to inspect.

---
### 📄 Markdown Documents Used
- `javascript_tutorial.md` — JavaScript Tutorial
- `blockchain.md` — Blockchain for Beginners
- `fortinet.md` — Fortinet FortiGate Security Target

> **Prerequisites:** Download the three sample PDFs linked in the root README using the expected filenames, run the main ingestion notebook to create `markdown_docs`, `parent_store`, and `qdrant_db`, and pull both configured Ollama models. Standard hosted Colab does not provide Ollama, so this evaluation is local-first unless you adapt both answer and judge models.


## 1. Install Dependencies

In [ ]:
import os
import subprocess

IN_COLAB_SETUP = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB_SETUP and not os.path.exists("requirements.txt"):
    repo_dir = "agentic-rag-for-dummies"
    if not os.path.isdir(repo_dir):
        subprocess.run(["git", "clone", "https://github.com/GiovanniPasq/agentic-rag-for-dummies.git"], check=True)
    os.chdir(repo_dir)

REQ_PATH = "requirements.txt" if os.path.exists("requirements.txt") else "../requirements.txt"
!pip install -qU -r {REQ_PATH}

## 2. Imports & Configuration

The app answer model generates the response; the RAGAS judge model scores it. Keeping them different reduces self-grading bias and gives a more independent signal. It does not make the score objective, but it avoids asking the same model to grade its own answer.

RAGAS collection metrics require a modern structured-output LLM. Ollama exposes an OpenAI-compatible API, so the judge stays local while RAGAS talks to it through the OpenAI client.


In [ ]:
import os
import sys
import ast
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm


# RAGAS 0.4.3 imports a legacy path during startup.
# The guard only prevents that import failure; evaluation still uses local Ollama.
import types

try:
    import langchain_community.chat_models.vertexai  # noqa: F401
except ModuleNotFoundError:
    vertexai_module = types.ModuleType("langchain_community.chat_models.vertexai")

    class ChatVertexAI:
        pass

    vertexai_module.ChatVertexAI = ChatVertexAI
    sys.modules["langchain_community.chat_models.vertexai"] = vertexai_module

from ragas.metrics.collections import (
    AnswerAccuracy,
    ContextRelevance,
    ResponseGroundedness,
    ContextPrecision,
    ContextRecall,
)
from openai import AsyncOpenAI
from ragas.llms import llm_factory

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ
PROJECT_ROOT = "project" if IN_COLAB else "../project"  # adjust if needed
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import config as project_config

# Keep the judge separate from the app answer model to reduce self-evaluation bias.
ANSWER_MODEL = project_config.LLM_MODEL
JUDGE_MODEL = project_config.JUDGE_MODEL
ollama_client = AsyncOpenAI(
    base_url="http://localhost:11434/v1/",
    api_key="ollama",  # required by the client, ignored by Ollama
)
ragas_llm = llm_factory(
    JUDGE_MODEL,
    client=ollama_client,
    temperature=0,
)

print(f"✅ Answer model under test: {ANSWER_MODEL}")
print(f"✅ RAGAS judge model: {JUDGE_MODEL}")

## 3. Import the RAG System

We import directly from the `rag_agent` package — no need to have the main notebook running in the same kernel.

**Project structure expected:**
```
project/
├── rag_agent/
│   ├── __init__.py
│   ├── graph.py          ← create_agent_graph()
│   ├── graph_state.py
│   ├── nodes.py
│   ├── edges.py
│   ├── tools.py          ← ToolFactory
│   ├── prompts.py
│   └── schemas.py
├── db/
│   └── parent_store_manager.py
├── config.py             ← LLM, embeddings, Qdrant client, paths
└── utils.py
```

> Adjust `PROJECT_ROOT` if you run this notebook from a different directory.

In [ ]:
from core.rag_system import RAGSystem

rag = RAGSystem()
rag.initialize()

agent_graph = rag.agent_graph

print("✅ RAG system ready")

### 3b. `query_rag` helper

Runs a single question through the full agentic pipeline and returns the answer together
with the actual child and parent retrieval outputs used by the agent.

In [ ]:
from langchain_core.messages import HumanMessage

def query_rag(question: str) -> dict:
    rag.reset_thread()   # evaluate each query independently
    config = rag.get_config()
    result = agent_graph.invoke({"messages": [HumanMessage(content=question)]}, config)
    answer = result["messages"][-1].content

    contexts = []
    for agent_answer in sorted(result.get("agent_answers", []), key=lambda item: item["index"]):
        contexts.extend(agent_answer.get("contexts", []))
    contexts = list(dict.fromkeys(contexts))

    return {"answer": answer, "contexts": contexts}

## 4. QA Evaluation Dataset

This section creates a compact evaluation dataset from `notebooks/data/curated_ragas_qa.json`.

Each record contains:

- `question`: what the user asks
- `reference`: the expected human-written answer
- `markdown_file`: the source document
- `context_phrases`: short anchors used to extract the reference context from Markdown

The final QA set contains **30 single-hop pairs**: 10 from each Markdown document. Keeping the data in JSON makes the notebook cleaner and lets you inspect or improve the evaluation set without touching the notebook logic.


In [ ]:
MARKDOWN_DIR_CANDIDATES = [Path("markdown_docs"), Path("../markdown_docs")]
MARKDOWN_DIR = next((path for path in MARKDOWN_DIR_CANDIDATES if path.exists()), None)
if MARKDOWN_DIR is None:
    searched = ", ".join(str(path.resolve()) for path in MARKDOWN_DIR_CANDIDATES)
    raise FileNotFoundError(f"No markdown_docs directory found. Searched: {searched}")

md_paths = sorted(MARKDOWN_DIR.glob("*.md"))
if not md_paths:
    raise FileNotFoundError(f"No markdown files found in {MARKDOWN_DIR.resolve()}")

doc_texts = {path.name: path.read_text(encoding="utf-8") for path in md_paths}

required_docs = {"javascript_tutorial.md", "blockchain.md", "fortinet.md"}
missing_docs = required_docs - set(doc_texts)
if missing_docs:
    raise FileNotFoundError(f"Missing expected markdown files: {sorted(missing_docs)}")


def extract_context(markdown_file: str, phrases: list[str], window: int = 1400) -> str:
    """Return compact source-grounded context around every matching phrase."""
    text = doc_texts[markdown_file]
    lower_text = text.lower()
    spans = []

    for phrase in phrases:
        position = lower_text.find(phrase.lower())
        if position < 0:
            continue
        start = max(0, position - window // 3)
        end = min(len(text), position + len(phrase) + window)
        spans.append((start, end))

    if not spans:
        return text[:window].strip()

    spans.sort()
    merged_spans = []
    for start, end in spans:
        if merged_spans and start <= merged_spans[-1][1]:
            merged_spans[-1] = (merged_spans[-1][0], max(merged_spans[-1][1], end))
        else:
            merged_spans.append((start, end))

    return "\n\n...\n\n".join(text[start:end].strip() for start, end in merged_spans)


qa_path_candidates = [
    Path("notebooks/data/curated_ragas_qa.json"),
    Path("data/curated_ragas_qa.json"),
]
qa_path = next((path for path in qa_path_candidates if path.exists()), None)
if qa_path is None:
    searched = ", ".join(str(path.resolve()) for path in qa_path_candidates)
    raise FileNotFoundError(f"No QA dataset found. Searched: {searched}")

curated_records = json.loads(qa_path.read_text(encoding="utf-8"))
for record in curated_records:
    record["reference_contexts"] = [
        extract_context(record["markdown_file"], record["context_phrases"])
    ]

curated_df = pd.DataFrame(curated_records)
curated_df.to_csv("ragas_qa_testset.csv", index=False)

print(f"Loaded {len(md_paths)} markdown files.")
print(f"✅ QA records ready: {len(curated_records)} samples")
print("✅ Saved to ragas_qa_testset.csv")

### 4b. Inspect the QA Dataset

In [ ]:
print(curated_df.shape)
try:
    display(curated_df[["source", "question", "reference"]])
except NameError:
    print(curated_df[["source", "question", "reference"]].to_string(index=False))

## 5. Run the RAG Pipeline on QA Questions

> ⚠️ This cell queries the RAG once per QA question. It may take several minutes depending on your local LLM.

If `ragas_evaluation_dataset.csv` already exists and still matches the current curated QA dataset, you can skip this section. Rerun it after changing the QA data, models, prompts, chunking, or retrieval settings.


In [ ]:
if "curated_records" not in globals():
    raise RuntimeError("Run section 4 first to load curated_records.")

results = []

for record in tqdm(curated_records, total=len(curated_records), desc="Querying RAG"):
    question = record["question"]
    reference = record["reference"]
    reference_contexts = record["reference_contexts"]
    source = record["source"]

    try:
        rag_output = query_rag(question)
        results.append({
            "source": source,
            "question": question,
            "answer": rag_output["answer"],
            "contexts": rag_output["contexts"],
            "ground_truth": reference,
            "reference_contexts": reference_contexts,
            "run_error": "",
        })
    except Exception as e:
        print(f"❌ Error for: '{question[:60]}...' — {e}")
        results.append({
            "source": source,
            "question": question,
            "answer": "",
            "contexts": [],
            "ground_truth": reference,
            "reference_contexts": reference_contexts,
            "run_error": str(e),
        })

print(f"\n✅ Done. {len(results)} results collected.")

## 6. Load or Prepare Saved RAG Outputs

The metrics need the question, generated answer, actual retrieved contexts, and reference answer. Pipeline failures are stored in `run_error` and skipped by the metric loop rather than being treated as ordinary low-quality answers.


In [ ]:
def parse_context_list(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else [value]
        except (SyntaxError, ValueError):
            return [value] if value.strip() else []
    return []


def expected_dataset_frame(records):
    return pd.DataFrame(
        {
            "source": record["source"],
            "user_input": record["question"],
            "reference": record["reference"],
        }
        for record in records
    )


def assert_saved_outputs_match_dataset(saved_df, records):
    expected_df = expected_dataset_frame(records).reset_index(drop=True)
    actual_df = saved_df[["source", "user_input", "reference"]].reset_index(drop=True)
    if not actual_df.equals(expected_df):
        raise RuntimeError(
            "Saved ragas_evaluation_dataset.csv does not match the current curated QA dataset. "
            "Run section 5 again to regenerate RAG answers before scoring."
        )


evaluation_csv_path = Path("./ragas_evaluation_dataset.csv")

if "curated_records" not in globals():
    raise RuntimeError("Run section 4 first to load the current curated QA dataset.")

if globals().get("results"):
    ragas_df = pd.DataFrame(
        {
            "source": r.get("source", "unknown"),
            "user_input": r["question"],
            "retrieved_contexts": r["contexts"],
            "reference_contexts": r.get("reference_contexts", []),
            "response": r["answer"],
            "reference": r["ground_truth"],
            "run_error": r.get("run_error", ""),
        }
        for r in results
    )
    assert_saved_outputs_match_dataset(ragas_df, curated_records)
    ragas_df.to_csv(evaluation_csv_path, index=False)
    print(f"✅ Saved fresh RAG outputs to {evaluation_csv_path}")
else:
    if not evaluation_csv_path.exists():
        raise RuntimeError(
            "No fresh results and no saved ragas_evaluation_dataset.csv found. Run section 5 first to query the RAG."
        )

    print(f"✅ Loading saved RAG outputs from {evaluation_csv_path}")
    ragas_df = pd.read_csv(evaluation_csv_path)
    assert_saved_outputs_match_dataset(ragas_df, curated_records)
    if "run_error" not in ragas_df.columns:
        ragas_df["run_error"] = ""
    ragas_df["run_error"] = ragas_df["run_error"].fillna("")
    ragas_df["retrieved_contexts"] = ragas_df["retrieved_contexts"].apply(parse_context_list)
    ragas_df["reference_contexts"] = ragas_df["reference_contexts"].apply(parse_context_list)

    results = [
        {
            "source": row.get("source", "saved_csv"),
            "question": row["user_input"],
            "answer": row["response"],
            "ground_truth": row["reference"],
            "contexts": row["retrieved_contexts"],
            "reference_contexts": row["reference_contexts"],
            "run_error": row.get("run_error", ""),
        }
        for _, row in ragas_df.iterrows()
    ]

print(f"✅ Evaluation rows ready: {len(ragas_df)}")


## 7. Compute RAGAS Metrics

### Metric definitions

| Metric | Family | Description | Range |
|---|---|---|---|
| **Answer Accuracy** | NVIDIA metric | Agreement between generated answer and reference answer | 0–1 |
| **Context Relevance** | NVIDIA metric | Relevance of retrieved contexts to the query | 0–1 |
| **Response Groundedness** | NVIDIA metric | Support of response claims in retrieved contexts | 0–1 |
| **Context Precision** | Retrieval metric | Whether relevant chunks are ranked higher than irrelevant chunks | 0–1 |
| **Context Recall** | Retrieval metric | Whether retrieved contexts cover the reference answer | 0–1 |

Each scorer follows the RAGAS collection-metric examples: create the metric, call `await scorer.ascore(...)`, store the score, and average at the end.


In [ ]:
metric_scorers = {
    "answer_accuracy": AnswerAccuracy(llm=ragas_llm),
    "context_relevance": ContextRelevance(llm=ragas_llm),
    "response_groundedness": ResponseGroundedness(llm=ragas_llm),
    "context_precision": ContextPrecision(llm=ragas_llm),
    "context_recall": ContextRecall(llm=ragas_llm),
}


async def score_answer(row, scorers):
    if row.get("run_error"):
        tqdm.write(f"   skipped pipeline failure: {row['run_error']}")
        return {metric_name: np.nan for metric_name in scorers}

    retrieved_contexts = row["retrieved_contexts"]

    metric_inputs = {
        "answer_accuracy": {
            "user_input": row["user_input"],
            "response": row["response"],
            "reference": row["reference"],
        },
        "context_relevance": {
            "user_input": row["user_input"],
            "retrieved_contexts": retrieved_contexts,
        },
        "response_groundedness": {
            "response": row["response"],
            "retrieved_contexts": retrieved_contexts,
        },
        "context_precision": {
            "user_input": row["user_input"],
            "reference": row["reference"],
            "retrieved_contexts": retrieved_contexts,
        },
        "context_recall": {
            "user_input": row["user_input"],
            "retrieved_contexts": retrieved_contexts,
            "reference": row["reference"],
        },
    }

    scores = {}
    for metric_name, scorer in scorers.items():
        try:
            result = await scorer.ascore(**metric_inputs[metric_name])
            scores[metric_name] = result.value
            tqdm.write(f"   {metric_name}: {result.value:.3f}")
        except Exception as exc:
            scores[metric_name] = np.nan
            tqdm.write(f"   {metric_name}: failed ({exc})")

    return scores


print("🔄 Running RAGAS metrics...")
score_rows = []
for idx, row in tqdm(ragas_df.iterrows(), total=len(ragas_df), desc="Evaluating"):
    tqdm.write(f"\n[{idx + 1}/{len(ragas_df)}] {row['user_input'][:90]}")
    score_rows.append(await score_answer(row, metric_scorers))

scores_df = pd.DataFrame(score_rows)
print("\n✅ Done!")
print(scores_df.mean(numeric_only=True).round(3).to_string())
scores_df.head()

## 8. Results as a DataFrame

In [ ]:
df = pd.concat([ragas_df.reset_index(drop=True), scores_df], axis=1)

if "source" not in df.columns:
    df.insert(0, "source", [r["source"] for r in results])
df["question"] = df["user_input"]
df["answer"] = df["response"]
df["ground_truth"] = df["reference"]

metric_cols = [
    "answer_accuracy",
    "context_relevance",
    "response_groundedness",
    "context_precision",
    "context_recall",
]

missing_metric_cols = [col for col in metric_cols if col not in df.columns]
if missing_metric_cols:
    print(f"⚠️ Missing metric columns: {missing_metric_cols}")

metric_cols = [col for col in metric_cols if col in df.columns]
df.head()

## 9. Aggregate Scores per Source

In [ ]:
agg = df.groupby("source")[metric_cols].mean().round(3)
agg.loc["OVERALL"] = df[metric_cols].mean().round(3)

print("=" * 70)
print(" RAGAS Scores — Mean per Source")
print("=" * 70)
print(agg.to_string())
print("=" * 70)

## 10. Visualisations

In [ ]:
# ── 10A. Heatmap + Grouped bar chart ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# Heatmap
sns.heatmap(
    agg.drop("OVERALL"),
    ax=axes[0],
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    linewidths=0.5,
    cbar_kws={"label": "Score"},
)
axes[0].set_title("RAGAS Metrics — Mean Score per Source", fontsize=13, fontweight="bold")
axes[0].tick_params(axis="x", rotation=20)

# Grouped bar chart
sources = agg.drop("OVERALL").index.tolist()
x = range(len(metric_cols))
width = 0.8 / max(len(sources), 1)
colors = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2", "#59a14f"]

for i, src in enumerate(sources):
    vals = agg.loc[src, metric_cols].values
    axes[1].bar(
        [xi + i * width for xi in x],
        vals,
        width=width,
        label=src,
        color=colors[i % len(colors)],
        alpha=0.85,
        edgecolor="white",
    )

axes[1].axhline(
    y=agg.loc["OVERALL", metric_cols].mean(),
    color="gray",
    linestyle="--",
    linewidth=1,
    label="Overall avg",
)
axes[1].set_xticks([xi + width * max(len(sources) - 1, 0) / 2 for xi in x])
axes[1].set_xticklabels([c.replace("_", "\n") for c in metric_cols], fontsize=9)
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Score")
axes[1].set_title("RAGAS Metrics — Comparison per Source", fontsize=13, fontweight="bold")
axes[1].legend(loc="lower right", fontsize=8)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("rag_evaluation_heatmap_bar.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Chart saved to rag_evaluation_heatmap_bar.png")

In [ ]:
# ── 10B. Radar chart — Overall profile ───────────────────────────────────────
labels       = [c.replace("_", "\n") for c in metric_cols]
overall_vals = agg.loc["OVERALL", metric_cols].values.tolist()
overall_vals += overall_vals[:1]   # close the polygon

angles = np.linspace(0, 2 * np.pi, len(metric_cols), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.plot(angles, overall_vals, color="steelblue", linewidth=2)
ax.fill(angles, overall_vals, color="steelblue", alpha=0.25)
ax.set_thetagrids(np.degrees(angles[:-1]), labels, fontsize=10)
ax.set_ylim(0, 1)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7)
ax.set_title("Overall RAGAS Profile", fontsize=13, fontweight="bold", pad=20)
ax.grid(color="grey", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("rag_evaluation_radar.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Radar chart saved to rag_evaluation_radar.png")

## 11. Save Results to CSV

In [ ]:
output_path = "rag_evaluation_results.csv"
df[["source", "question", "answer", "ground_truth", "run_error"] + metric_cols].to_csv(
    output_path, index=False
)
print(f"✅ Full results saved to {output_path}")

agg.to_csv("rag_evaluation_summary.csv")
print("✅ Summary saved to rag_evaluation_summary.csv")

## 12. Interpretation Guide

```
┌───────────────────────────────────────────────────────────────────────────┐
│  Metric                  │ Score < 0.4   │ 0.4-0.7 (OK) │ > 0.7 (Good)   │
│───────────────────────────────────────────────────────────────────────────│
│  Answer Accuracy         │ Wrong answer  │ Partial match│ Matches ref     │
│  Context Relevance       │ Off-topic ctx │ Some focus   │ Focused context │
│  Response Groundedness   │ Unsupported   │ Partly based │ Fully grounded  │
│  Context Precision       │ Noisy rank    │ Some noise   │ Relevant first  │
│  Context Recall          │ Missing info  │ Partial info │ Complete info   │
└───────────────────────────────────────────────────────────────────────────┘
```

### How to improve each metric

| Low metric | Likely cause | Fix |
|---|---|---|
| Answer Accuracy | Generated answer does not match the reference | Improve prompts, retrieval coverage, or use a stronger answer model |
| Context Relevance | Retrieved chunks are off-topic | Increase `score_threshold`, improve query rewriting, or tune embeddings |
| Response Groundedness | Answer includes claims not supported by retrieved chunks | Strengthen grounding instructions and reduce unsupported synthesis |
| Context Precision | Relevant chunks are not ranked first | Reduce `k`, increase `score_threshold`, or add reranking |
| Context Recall | Required information was not retrieved | Lower `score_threshold`, increase `k`, improve embeddings, or split documents differently |

### Agentic metrics available in RAGAS, not implemented here

RAGAS also provides [agentic/tool-use metrics](https://docs.ragas.io/en/latest/concepts/metrics/available_metrics/agents/) for workflows where you want to judge the agent trace rather than only the final RAG answer:

| Metric | Use when |
|---|---|
| `ToolCallAccuracy` | You have expected tool calls and need exact sequence/argument matching |
| `ToolCallF1` | You want softer precision/recall scoring over tool calls |
| `AgentGoalAccuracyWithReference` | You have a reference final outcome for the agent task |
| `AgentGoalAccuracyWithoutReference` | You want the judge to decide whether the goal was achieved without a reference |
| `TopicAdherence` | You need to verify whether a conversation stays within expected topics |

They are not implemented in this notebook because they require multi-turn traces, expected tool calls, or goal-state annotations. This notebook evaluates the final RAG answer and retrieved contexts with single-turn RAG metrics only.

### How the agent graph affects scores

| Component | Metric impacted | Tuning lever |
|---|---|---|
| `search_child_chunks` `score_threshold` | Context Precision / Recall / Relevance | Raise = more precise, lower = more recall |
| `MAX_TOOL_CALLS` / `MAX_ITERATIONS` | Answer Accuracy / Context Recall | Higher = more research, slower |
| `compress_context` | Response Groundedness / Answer Accuracy | Compression can drop details needed by the final answer |
| Query rewriting | Context Relevance / Context Recall | Better rewrites improve retrieval coverage and focus |